# OFF Ontology Data Cleaning — without Deduplication

## Goal
This notebook cleans the `off_data_ontology.csv` dataset (OpenFoodFacts, with L1/L2/L3 ontology labels) following the same pipeline as `eda2.ipynb` — **without deduplication via fuzzy matching**.

---

## Input File
| Attribute | Value |
|---|---|
| File | `off_data_ontology.csv` |
| Rows (raw) | 40,996 |
| Columns | `item_name`, `cat`, `kcal_100g`, `fat_100g`, `carbs_100g`, `protein_100g`, `source`, `cat_l1`, `cat_l2`, `cat_l3` |

---

## Cleaning Steps

### Step 1 — Load
Read the dataset using `pandas.read_csv()`.

### Step 2 — Ensure `item_name` is a string
- `.astype(str)` + `.str.strip()` applied to `item_name`
- Rows where `item_name` becomes `"nan"` or `""` are removed

### Step 3 — Lowercase & normalise hyphens
- All `item_name` values are converted to lowercase
- Hyphens (`-`) are replaced with spaces

### Step 4 — Word count filter (≤ 5 words)
- `item_name` entries with more than 5 words are removed
- Rationale: too many words indicate descriptive text, ingredient lists or malformed entries
- **Dropped: 6,984 rows**

### Step 5 — Remove measurement keywords
- Entries containing measurement keywords are filtered out (regex-based, word boundary `\b`)
- Categories: weight/mass, volume, count/portion, packaging, bakery, recipe amounts (> 100 keywords)
- **Dropped: 4,289 rows**

### Step 6 — Remove entries containing digits
- Rows with digits in `item_name` are removed (regex `\d`)
- Rationale: product codes, gram values or SKUs are not valid food names
- **Dropped: 1,573 rows**

### Step 7 — Unicode normalisation & special characters
- Unicode NFKD → ASCII (removes accents, umlauts, etc.)
- `&` → `" and "`
- All non-alphanumeric characters are replaced with spaces
- Multiple spaces are collapsed (`.strip()`)
- Rows that are empty after normalisation are removed
- **Dropped: 130 rows**

---

## Output

| Attribute | Value |
|---|---|
| File | `off_data_ontology_clean.csv` |
| Rows (cleaned) | **28,020** |
| Unique `item_name` | 21,008 |
| Columns | `item_name`, `kcal_100g`, `fat_100g`, `carbs_100g`, `protein_100g`, `source`, `cat_l1`, `cat_l2` |

> **Not performed:** Fuzzy matching / deduplication (RapidFuzz BFS clustering).  
> This is intentionally left for a separate step.

## Step 1 — Load Data

In [4]:
import pandas as pd
import re
import unicodedata

df = pd.read_csv('off_data_ontology.csv', low_memory=False)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(df.columns.tolist())
df.head(3)

Loaded: 40,996 rows × 10 cols
['item_name', 'cat', 'kcal_100g', 'fat_100g', 'carbs_100g', 'protein_100g', 'source', 'cat_l1', 'cat_l2', 'cat_l3']


,item_name,cat,kcal_100g,fat_100g,carbs_100g,protein_100g,source,cat_l1,cat_l2,cat_l3
0,greek yogurt,greek-style yogurts,86.666667,1.666667,10.666667,7.333333,off,dairy & eggs,yogurt,greek-style yogurts
1,shake mix vanilla flavour,dietary supplements,376.000000,7.200000,42.000000,36.000000,off,supplements,dietary supplements,dietary supplements
2,german fine bread,breads,263.000000,3.510000,52.630000,7.020000,off,grains & pasta,bread,breads


## Step 2 — Ensure `item_name` is a string & strip whitespace

In [5]:
# Cast to string and strip
df['item_name'] = df['item_name'].astype(str).str.strip()

# Drop rows where item_name ended up empty or 'nan'
before = len(df)
df = df[df['item_name'].str.lower() != 'nan']
df = df[df['item_name'] != '']
print(f"Dropped {before - len(df):,} null/empty item_name rows → {len(df):,} remaining")

Dropped 0 null/empty item_name rows → 40,996 remaining


## Step 3 — Lowercase & normalise hyphens

In [6]:
df['item_name'] = df['item_name'].str.lower().str.replace('-', ' ', regex=False)
print(df['item_name'].head(5).tolist())

['greek yogurt', 'shake mix vanilla flavour', 'german fine bread', 'harvest whole wheat bread', 'sourdough bread']


## Step 4 — Filter: keep `item_name` ≤ 5 words

In [7]:
df['item_name_word_count'] = df['item_name'].str.split().apply(len)
before = len(df)
df = df[df['item_name_word_count'] <= 5]
print(f"Dropped {before - len(df):,} rows with >5-word item names → {len(df):,} remaining")
print(df['item_name_word_count'].value_counts().sort_index())

Dropped 6,984 rows with >5-word item names → 34,012 remaining
item_name_word_count
1    3258
2    9042
3    9667
4    7443
5    4602
Name: count, dtype: int64


## Step 5 — Remove measurement keywords

In [8]:
food_measurements = [
    # Weight / mass
    "mg", "milligram", "milligrams",
    "g", "gram", "grams",
    "kg", "kilogram", "kilograms",
    "oz", "ounce", "ounces",
    "lb", "lbs", "pound", "pounds",
    # Volume
    "ml", "milliliter", "milliliters", "millilitre", "millilitres",
    "cl", "centiliter", "centiliters", "centilitre", "centilitres",
    "l", "liter", "liters", "litre", "litres",
    "tsp", "teaspoon", "teaspoons",
    "tbsp", "tablespoon", "tablespoons",
    "fl oz", "fluid ounce", "fluid ounces",
    "cup", "cups",
    "pint", "pints",
    "quart", "quarts",
    "gallon", "gallons",
    "dash", "dashes",
    "pinch", "pinches",
    "splash", "splashes",
    "drop", "drops",
    # Count / piece-based
    "piece", "pieces",
    "pc", "pcs",
    "unit", "units",
    "item", "items",
    "whole", "halves", "half",
    "quarter", "quarters",
    "slice", "slices",
    "stick", "sticks",
    "cube", "cubes",
    "chunk", "chunks",
    "wedge", "wedges",
    "strip", "strips",
    "ring", "rings",
    "clove", "cloves",
    "leaf", "leaves",
    "sprig", "sprigs",
    "stalk", "stalks",
    "stem", "stems",
    "head", "heads",
    "bunch", "bunches",
    "bulb", "bulbs",
    "ear", "ears",
    "kernel", "kernels",
    "pod", "pods",
    "bean", "beans",
    "egg", "eggs",
    "fillet", "fillets",
    "breast", "breasts",
    "thigh", "thighs",
    "drumstick", "drumsticks",
    "leg", "legs",
    "wing", "wings",
    # Serving style
    "serving", "servings",
    "portion", "portions",
    "helping", "helpings",
    "plate", "plates",
    "bowl", "bowls",
    "dish", "dishes",
    "tray", "trays",
    # Packaging / container-based
    "pack", "packs",
    "packet", "packets",
    "package", "packages",
    "bag", "bags",
    "box", "boxes",
    "carton", "cartons",
    "can", "cans",
    "tin", "tins",
    "jar", "jars",
    "bottle", "bottles",
    "tube", "tubes",
    "sachet", "sachets",
    "wrapper", "wrappers",
    "container", "containers",
    "cupful", "cupfuls",
    # Bakery / produce / retail
    "loaf", "loaves",
    "roll", "rolls",
    "bun", "buns",
    "patty", "patties",
    "link", "links",
    "sausage", "sausages",
    "ball", "balls",
    "bar", "bars",
    "block", "blocks",
    # Recipe amounts
    "handful", "handfuls",
    "fistful", "fistfuls",
    "scoop", "scoops",
    "ladle", "ladles",
    "spoonful", "spoonfuls",
    "heaped teaspoon", "heaped teaspoons",
    "heaped tablespoon", "heaped tablespoons",
    "level teaspoon", "level teaspoons",
    "level tablespoon", "level tablespoons",
    "to taste",
]

pattern = r'\b(?:' + '|'.join(re.escape(m) for m in food_measurements) + r')\b'
before = len(df)
df = df[~df['item_name'].str.contains(pattern, regex=True)]
print(f"Dropped {before - len(df):,} rows containing measurement keywords → {len(df):,} remaining")

Dropped 4,289 rows containing measurement keywords → 29,723 remaining


## Step 6 — Remove entries containing digits

In [9]:
before = len(df)
df = df[~df['item_name'].str.contains(r'\d', regex=True)]
print(f"Dropped {before - len(df):,} rows with digits in item_name → {len(df):,} remaining")

Dropped 1,573 rows with digits in item_name → 28,150 remaining


## Step 7 — Normalise: Unicode → ASCII, remove special chars, collapse whitespace

In [10]:
def normalize_item_name(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = s.replace("&", " and ")
    s = re.sub(r"[^\w\s]", " ", s)          # remove special chars
    s = re.sub(r"\s+", " ", s).strip()      # collapse whitespace
    return s

df['item_name'] = df['item_name'].apply(normalize_item_name)

# Drop any rows that became empty after normalisation
before = len(df)
df = df[df['item_name'] != '']
print(f"Dropped {before - len(df):,} rows that became empty after normalisation → {len(df):,} remaining")
print(df['item_name'].head(10).tolist())

Dropped 130 rows that became empty after normalisation → 28,020 remaining
['greek yogurt', 'shake mix vanilla flavour', 'german fine bread', 'sourdough bread', 'cracked wheat sourdough bread', 'seriously salt and vinegar', 'bakers best white bread', 'bakers best rye bread', 'gluten free multigrain bread', 'salami de dinde']


## Step 8 — Summary & Save

In [11]:
print("=== Final Dataset ===")
print(f"Rows : {len(df):,}")
print(f"Cols : {df.columns.tolist()}")
print(f"Unique item_names: {df['item_name'].nunique():,}")
print()
print(df.dtypes)
df.head(5)

=== Final Dataset ===
Rows : 28,020
Cols : ['item_name', 'cat', 'kcal_100g', 'fat_100g', 'carbs_100g', 'protein_100g', 'source', 'cat_l1', 'cat_l2', 'cat_l3', 'item_name_word_count']
Unique item_names: 21,008

item_name                   str
cat                         str
kcal_100g               float64
fat_100g                float64
carbs_100g              float64
protein_100g            float64
source                      str
cat_l1                      str
cat_l2                      str
cat_l3                      str
item_name_word_count      int64
dtype: object


,item_name,cat,kcal_100g,fat_100g,carbs_100g,protein_100g,source,cat_l1,cat_l2,cat_l3,item_name_word_count
0,greek yogurt,greek-style yogurts,86.666667,1.666667,10.666667,7.333333,off,dairy & eggs,yogurt,greek-style yogurts,2
1,shake mix vanilla flavour,dietary supplements,376.000000,7.200000,42.000000,36.000000,off,supplements,dietary supplements,dietary supplements,4
2,german fine bread,breads,263.000000,3.510000,52.630000,7.020000,off,grains & pasta,bread,breads,3
4,sourdough bread,sliced breads,214.285714,1.403875,59.118050,8.390992,off,grains & pasta,bread,sliced breads,2
5,cracked wheat sourdough bread,sourdough breads,211.538462,0.961538,46.153846,7.692308,off,grains & pasta,bread,sourdough breads,4


In [12]:
df.to_csv('off_data_ontology_clean.csv', index=False)
print("Saved → off_data_ontology_clean.csv")

Saved → off_data_ontology_clean.csv


In [13]:
#show the entry for "banana"
df[df['item_name'] == 'banane']

,item_name,cat,kcal_100g,fat_100g,carbs_100g,protein_100g,source,cat_l1,cat_l2,cat_l3,item_name_word_count
920,banane,bananas,161.0,8.2,0.8,21.0,off,fruits,fresh fruits,bananas,1


In [14]:
#drop cat_l3 in the dataframe
df.drop(columns=["cat_l3"])

,item_name,cat,kcal_100g,fat_100g,carbs_100g,protein_100g,source,cat_l1,cat_l2,item_name_word_count
0,greek yogurt,greek-style yogurts,86.666667,1.666667,10.666667,7.333333,off,dairy & eggs,yogurt,2
1,shake mix vanilla flavour,dietary supplements,376.000000,7.200000,42.000000,36.000000,off,supplements,dietary supplements,4
2,german fine bread,breads,263.000000,3.510000,52.630000,7.020000,off,grains & pasta,bread,3
4,sourdough bread,sliced breads,214.285714,1.403875,59.118050,8.390992,off,grains & pasta,bread,2
5,cracked wheat sourdough bread,sourdough breads,211.538462,0.961538,46.153846,7.692308,off,grains & pasta,bread,4
...,...,...,...,...,...,...,...,...,...,...
40990,rhubarb yoghurt,yogurts,105.000000,3.300000,15.200000,3.400000,off,dairy & eggs,yogurt,2
40991,fde nutrition,nuts and their products,352.000000,16.300000,40.800000,5.800000,off,snacks,nuts & seeds,2
40992,mayonnaise de dijon,groceries,0.000000,70.000000,2.000000,1.200000,off,other,undefined,3
40993,casabe paul,breads,353.000000,0.000000,87.060000,1.180000,off,grains & pasta,bread,2


In [15]:
#drop item_name_word_count
df.drop(columns=["item_name_word_count"])

,item_name,cat,kcal_100g,fat_100g,carbs_100g,protein_100g,source,cat_l1,cat_l2,cat_l3
0,greek yogurt,greek-style yogurts,86.666667,1.666667,10.666667,7.333333,off,dairy & eggs,yogurt,greek-style yogurts
1,shake mix vanilla flavour,dietary supplements,376.000000,7.200000,42.000000,36.000000,off,supplements,dietary supplements,dietary supplements
2,german fine bread,breads,263.000000,3.510000,52.630000,7.020000,off,grains & pasta,bread,breads
4,sourdough bread,sliced breads,214.285714,1.403875,59.118050,8.390992,off,grains & pasta,bread,sliced breads
5,cracked wheat sourdough bread,sourdough breads,211.538462,0.961538,46.153846,7.692308,off,grains & pasta,bread,sourdough breads
...,...,...,...,...,...,...,...,...,...,...
40990,rhubarb yoghurt,yogurts,105.000000,3.300000,15.200000,3.400000,off,dairy & eggs,yogurt,yogurts
40991,fde nutrition,nuts and their products,352.000000,16.300000,40.800000,5.800000,off,snacks,nuts & seeds,nuts and their products
40992,mayonnaise de dijon,groceries,0.000000,70.000000,2.000000,1.200000,off,other,undefined,groceries
40993,casabe paul,breads,353.000000,0.000000,87.060000,1.180000,off,grains & pasta,bread,breads


In [16]:
df = df.drop(columns=["cat","cat_l3","item_name_word_count"])

In [17]:
df

,item_name,kcal_100g,fat_100g,carbs_100g,protein_100g,source,cat_l1,cat_l2
0,greek yogurt,86.666667,1.666667,10.666667,7.333333,off,dairy & eggs,yogurt
1,shake mix vanilla flavour,376.000000,7.200000,42.000000,36.000000,off,supplements,dietary supplements
2,german fine bread,263.000000,3.510000,52.630000,7.020000,off,grains & pasta,bread
4,sourdough bread,214.285714,1.403875,59.118050,8.390992,off,grains & pasta,bread
5,cracked wheat sourdough bread,211.538462,0.961538,46.153846,7.692308,off,grains & pasta,bread
...,...,...,...,...,...,...,...,...
40990,rhubarb yoghurt,105.000000,3.300000,15.200000,3.400000,off,dairy & eggs,yogurt
40991,fde nutrition,352.000000,16.300000,40.800000,5.800000,off,snacks,nuts & seeds
40992,mayonnaise de dijon,0.000000,70.000000,2.000000,1.200000,off,other,undefined
40993,casabe paul,353.000000,0.000000,87.060000,1.180000,off,grains & pasta,bread


In [18]:
#load usda_final 
df_usda = pd.read_csv('usda_final.csv', low_memory=False)


In [19]:
df_usda

,item_name,carbs_100g,kcal_100g,fat_100g,protein_100g,source,cat_l1,cat_l2,cat_l3
0,beef broth,0.42,4.00,0.00,0.83,usda,condiments & sauces,seasoning & spices,herbs/spices/extracts
1,broth chicken,0.42,4.00,0.00,0.83,usda,condiments & sauces,seasoning & spices,herbs/spices/extracts
2,and bean campbells ham soup,10.61,61.00,0.61,3.67,usda,soups,prepared soup,prepared soups
3,campbells soup tomato,15.00,75.00,1.25,1.67,usda,soups,prepared soup,prepared soups
4,bread farm pepperidge tuscan,50.88,263.00,2.63,8.77,usda,grains & pasta,bread,bread
...,...,...,...,...,...,...,...,...,...
124120,bite grain pancake rich whole,43.81,438.00,27.67,4.61,usda,baked goods,cakes & pastries,sweet bakery products
124121,chicken dark ground tyson uncooked,0.00,164.62,10.78,16.90,usda,meat,processed meats & cold cuts,meat/poultry/other animals prepared/processed
124122,carbe diem orzo pasta,71.43,196.00,0.89,14.29,usda,grains & pasta,pasta & noodles,pasta/noodles
124123,large ny steak strip,0.00,223.00,15.18,20.54,usda,meat,processed meats & cold cuts,meat/poultry/other animals prepared/processed


In [20]:
df_usda = df_usda.drop(columns=["cat_l3"])

In [21]:
#concat both dataframes
df_full = pd.concat([df, df_usda], ignore_index=True)
print(f"Combined dataset: {len(df_full):,} rows × {df_full.shape[1]} cols")
df_full.head(5)

Combined dataset: 152,145 rows × 8 cols


,item_name,kcal_100g,fat_100g,carbs_100g,protein_100g,source,cat_l1,cat_l2
0,greek yogurt,86.666667,1.666667,10.666667,7.333333,off,dairy & eggs,yogurt
1,shake mix vanilla flavour,376.000000,7.200000,42.000000,36.000000,off,supplements,dietary supplements
2,german fine bread,263.000000,3.510000,52.630000,7.020000,off,grains & pasta,bread
3,sourdough bread,214.285714,1.403875,59.118050,8.390992,off,grains & pasta,bread
4,cracked wheat sourdough bread,211.538462,0.961538,46.153846,7.692308,off,grains & pasta,bread


In [22]:
#show NaN and Other null values in the combined dataframe
print("Null values in combined dataset:")
print(df_full.isnull().sum())

Null values in combined dataset:
item_name          0
kcal_100g          0
fat_100g           0
carbs_100g         0
protein_100g       0
source             0
cat_l1          2700
cat_l2          2700
dtype: int64


In [23]:
#drop all rows where cat_l1 and cat_l2 is "other" , "NaN" ,"babyfood"
df_full = df_full[~df_full['cat_l1'].isin(['other', 'NaN', 'babyfood'])]
df_full = df_full[~df_full['cat_l2'].isin(['other', 'NaN', 'babyfood'])]

In [24]:
print(df_full.isnull().sum())

item_name          0
kcal_100g          0
fat_100g           0
carbs_100g         0
protein_100g       0
source             0
cat_l1          2700
cat_l2          2700
dtype: int64


In [25]:
#drop rows with null values in cat_l1 and cat_l2
df_full = df_full.dropna(subset=['cat_l1', 'cat_l2'])

In [26]:
print(df_full.isnull().sum())


item_name       0
kcal_100g       0
fat_100g        0
carbs_100g      0
protein_100g    0
source          0
cat_l1          0
cat_l2          0
dtype: int64


In [27]:
#show cat_l2 "other"
df_full[df_full['cat_l2'] == 'other']

,item_name,kcal_100g,fat_100g,carbs_100g,protein_100g,source,cat_l1,cat_l2


In [28]:
#how many unique categories we have?    
print(f"Unique cat_l1: {df_full['cat_l1'].nunique():,}")
print(f"Unique cat_l2: {df_full['cat_l2'].nunique():,}")

Unique cat_l1: 19
Unique cat_l2: 85


In [29]:
#drop "baby food"
df_full = df_full[df_full['cat_l1'] != 'baby food']

In [30]:
df_full.cat_l1.unique()

<StringArray>
[            'dairy & eggs',              'supplements',
           'grains & pasta',                     'meat',
                   'snacks',                   'fruits',
  'prepared & frozen meals',                'beverages',
   'sweets & confectionery',              'baked goods',
               'vegetables', 'plant-based alternatives',
      'condiments & sauces',          'legumes & beans',
           'fish & seafood',              'fats & oils',
                    'soups',                  'poultry']
Length: 18, dtype: str

In [31]:
df_full.info()

<class 'pandas.DataFrame'>
Index: 148270 entries, 0 to 152144
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   item_name     148270 non-null  str    
 1   kcal_100g     148270 non-null  float64
 2   fat_100g      148270 non-null  float64
 3   carbs_100g    148270 non-null  float64
 4   protein_100g  148270 non-null  float64
 5   source        148270 non-null  str    
 6   cat_l1        148270 non-null  str    
 7   cat_l2        148270 non-null  str    
dtypes: float64(4), str(4)
memory usage: 10.2 MB


In [32]:
df_full.cat_l2.unique()

<StringArray>
[                     'yogurt',         'dietary supplements',
                       'bread', 'processed meats & cold cuts',
                'other snacks',                'fruit juices',
                  'snack bars',              'prepared meals',
                        'milk',                      'cheese',
                         'tea',              'chips & crisps',
                   'chocolate',               'candy & gummy',
              'jams & spreads',                    'crackers',
                'nuts & seeds',                'other grains',
           'frozen vegetables',            'plant-based meat',
              'cooking sauces',                      'cereal',
             'pasta & noodles',          'hummus & bean dips',
                  'fresh fish',           'ketchup & mustard',
            'fresh vegetables',          'butter & margarine',
              'butter & cream',            'cakes & pastries',
                'fresh fruits',          

In [33]:
#save and zip the full combined dataframe
df_full.to_csv('combined_final.csv', index=False)
print("Saved → combined_final.csv")

Saved → combined_final.csv


In [34]:
#zip the combined_final.csv
import zipfile

with zipfile.ZipFile('combined_final.zip', 'w') as zipf:
    zipf.write('combined_final.csv')
print("Zipped → combined_final.zip")

Zipped → combined_final.zip


In [35]:
df_full.info()

<class 'pandas.DataFrame'>
Index: 148270 entries, 0 to 152144
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   item_name     148270 non-null  str    
 1   kcal_100g     148270 non-null  float64
 2   fat_100g      148270 non-null  float64
 3   carbs_100g    148270 non-null  float64
 4   protein_100g  148270 non-null  float64
 5   source        148270 non-null  str    
 6   cat_l1        148270 non-null  str    
 7   cat_l2        148270 non-null  str    
dtypes: float64(4), str(4)
memory usage: 10.2 MB


# USDA Deduplication Pipeline (reference: `eda2.ipynb`)

## Goal
After the basic cleaning steps (lowercase, word-count filter, measurement removal, normalisation), the USDA dataset still contained a large number of near-duplicate `item_name` entries — e.g. `"apple"` vs `"apples"`, `"chicken breast"` vs `"chicken breasts"`. The deduplication pipeline collapses these into a single canonical name per cluster.

---

## Input
| Attribute | Value |
|---|---|
| File | cleaned USDA DataFrame (in-memory) |
| Column used | `item_norm` (sort-normalised version of `item_name`) |

### What is `item_norm`?
Before deduplication, each `item_name` is transformed into a **sort-normalised** form:
```
unicode → ASCII  →  lowercase  →  remove special chars  →  tokenise  →  deduplicate tokens  →  sort tokens alphabetically  →  rejoin
```
Example: `"Chicken, Breast"` → `item_norm = "breast chicken"`

This ensures that `"chicken breast"` and `"breast chicken"` map to the same normalised form and are caught as exact duplicates before even reaching the fuzzy step.

---

## Deduplication Steps

### Step 1 — Exact deduplication on `item_norm`
Drop all rows with a duplicate `item_norm`, keeping only one representative per unique normalised name:
```python
df_unique = df_usda.drop_duplicates(subset=["item_norm"])
```

### Step 2 — Build blocks (blocking)
To avoid comparing every name against every other name (O(n²) complexity), entries are grouped into **blocks** by a composite key:
```python
block_key = (first_4_chars_of_first_token, token_count, first_char_of_first_token)
```
Example: `"apple juice"` → block key `("appl", 2, "a")`

Only names within the same block are compared against each other. This dramatically reduces the number of comparisons required.

### Step 3 — Fuzzy matching within blocks (RapidFuzz)
Within each block, every pair of names is compared using **`fuzz.token_set_ratio`** from the `rapidfuzz` library:
- **Threshold:** score ≥ 95
- `token_set_ratio` is order-insensitive and handles subset relationships (e.g. `"red apple"` vs `"apple red"`)
- All pairs above the threshold are recorded as *similar pairs*

### Step 4 — Build clusters (BFS graph traversal)
The similar pairs form an undirected graph. Connected components are found via **Breadth-First Search (BFS)**:
- Each node is a unique `item_norm`
- An edge exists between two nodes if their fuzzy score ≥ 95
- Each connected component becomes one **cluster** of near-duplicate names

Example cluster:
```
- "apple"
- "apples"
- "apple fruit"
```

### Step 5 — Select a canonical name per cluster
For each cluster, one representative name is chosen as the **canonical name** using this priority:
1. **Highest frequency** in the original dataset (most common form)
2. **Shortest name** (as a tiebreaker — simpler is better)
3. **Alphabetical order** (final tiebreaker)

All other names in the cluster are mapped to this canonical name.

### Step 6 — Apply mapping & deduplicate
- The `canonical_map_fuzzy` is applied to the `item_norm` column → new column `item_canonical`
- Any name that was not part of any fuzzy cluster keeps itself as its own canonical name
- `drop_duplicates(subset=["item_canonical"])` removes all remaining duplicate rows

### Step 7 — Remove duplicate words within names
A post-processing pass removes any **repeated tokens** within a single `item_name`:
```python
"chicken chicken breast" → "chicken breast"
```
Applied to both `item_name` and `item_canonical`.

### Step 8 — Finalise columns
- `item_name` is replaced with the value of `item_canonical`
- Helper columns `item_norm` and `item_canonical` are dropped
- `item_name_word_count` is recalculated

---

## Output
| Attribute | Value |
|---|---|
| File | `usda_data_dedup_final.csv` |
| Library | `rapidfuzz` (`fuzz.token_set_ratio`) |
| Threshold | 95 |
| Strategy | Blocking + BFS clustering + frequency-based canonical selection |

---

## Why this approach?
| Design choice | Reason |
|---|---|
| Blocking before fuzzy | Avoids O(n²) comparisons on 130k+ entries |
| `token_set_ratio` over `ratio` | Order-insensitive; handles partial overlaps robustly |
| Threshold = 95 | High precision — avoids merging legitimately different foods (e.g. `"pork"` ≠ `"pork rinds"`) |
| BFS clustering | Transitivity — if A≈B and B≈C, all three collapse into one cluster |
| Frequency-first canonical | Preserves the most commonly used real-world name |